# 01 · Weather as a price driver

**Research question 1:** How, and by how much, does the weather change the Lithuanian day-ahead electricity price?

This notebook tests hypotheses H1–H5 on hourly data for January 2023 – August 2026
(the `lt_hourly` view built by `src/build_db.py`).

| Hypothesis | Statement |
|---|---|
| H1 | Stronger wind in the Baltic states lowers the Lithuanian day-ahead price, holding demand, solar, gas, Nordic hydro and calendar effects fixed. |
| H2 | The effect of Baltic wind on the price has grown from 2023 to 2026 as installed wind capacity increased. |
| H3 | Nordic and Continental wind also lower the Lithuanian price; without them the Baltic wind effect is overstated. |
| H4 | Wind lowers the price more when natural gas is expensive, because it displaces gas-fired plants. |
| H5 | More water than normal in Nordic hydro reservoirs lowers the Lithuanian price. |

**Identification strategy.** Wind is measured with ERA5 weather indices, not with actual generation:
the weather is not affected by the electricity market, while actual wind output is (wind farms are
curtailed when prices are negative). Common causes of wind and price (season, time of day, demand,
gas, hydro, regional weather) are controlled for. Mediators (imports, neighbouring prices, actual
generation) are deliberately left out, because controlling for them would block part of the effect.

**Robustness checks:** coefficient stability, a placebo test with future wind, a dose–response check,
a pre/post-synchronisation split, an instrumental-variable (2SLS) estimate and a median regression.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

HAC = {"cov_type": "HAC", "cov_kwds": {"maxlags": 48}}  # errors correlated over up to 48 hours
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

con = duckdb.connect(str(ROOT / "data" / "lt_power.duckdb"), read_only=True)
df = con.sql("SELECT * FROM lt_hourly ORDER BY ts_utc").df()
con.close()
print(f"{len(df):,} hours from {df['ts_local'].min()} to {df['ts_local'].max()}")

## 1. Variables and units

Units are chosen so that every coefficient reads as "€/MWh change in the hourly price".

| Variable | Meaning | Unit |
|---|---|---|
| `price` | Lithuanian day-ahead price | €/MWh |
| `wind_baltic` | Mean ERA5 wind capacity factor of Lithuania, Latvia and Estonia | % of capacity (0–100) |
| `wind_nordic` | The same for Sweden, Finland, Norway and Denmark | % of capacity |
| `wind_continental` | The same for Poland and Germany | % of capacity |
| `solar_rad` | ERA5 solar radiation in Lithuania | 100 W/m² |
| `load_fc_gw` | Day-ahead load forecast for Lithuania | GW |
| `gas` | Last TTF front-month close before the delivery day | €/MWh |
| `hydro_twh` | Nordic reservoir content minus the 2015–2022 normal for that week | TWh |

Fixed effects: hour of day, day of week, month and year (local Lithuanian time).

In [ ]:
df["year"] = df["ts_local"].dt.year
df["wind_lt"] = df["wind_cf_lt"] * 100
df["wind_lv_ee"] = df[["wind_cf_lv", "wind_cf_ee"]].mean(axis=1) * 100
df["wind_baltic"] = df[["wind_cf_lt", "wind_cf_lv", "wind_cf_ee"]].mean(axis=1) * 100
df["wind_nordic"] = df[["wind_cf_se", "wind_cf_fi", "wind_cf_no", "wind_cf_dk"]].mean(axis=1) * 100
df["wind_continental"] = df[["wind_cf_pl", "wind_cf_de"]].mean(axis=1) * 100
df["solar_rad"] = df["solar_rad_lt_wm2"] / 100
df["load_fc_gw"] = df["load_forecast_mw"] / 1000
df["gas"] = df["ttf_eur_mwh"]
df["hydro_twh"] = df["nordic_hydro_deviation_gwh"] / 1000
df["wind_mw_100"] = df["wind_mw"] / 100                       # actual wind output, per 100 MW
df["wind_baltic_lead168"] = df["wind_baltic"].shift(-168)     # wind one week later (placebo)

FE = "C(hour_local) + C(weekday) + C(month_local) + C(year)"
CONTROLS = f"solar_rad + load_fc_gw + gas + hydro_twh + {FE}"
MODEL_COLS = ["price", "wind_baltic", "wind_nordic", "wind_continental", "solar_rad",
              "load_fc_gw", "gas", "hydro_twh", "hour_local", "weekday", "month_local", "year"]

d = df.dropna(subset=MODEL_COLS).copy()
print(f"Model sample: {len(d):,} hours ({len(df) - len(d)} dropped because of missing values)")
d[["price", "wind_baltic", "wind_nordic", "wind_continental", "solar_rad", "load_fc_gw", "gas", "hydro_twh"]].describe().round(2)

In [ ]:
def effect(model, name, scale=1.0):
    #Coefficient of one variable with its 95% confidence interval and p-value, times `scale`
    ci = model.conf_int().loc[name] * scale
    return pd.Series({
        "effect": model.params[name] * scale,
        "ci_low": ci.min(),
        "ci_high": ci.max(),
        "p_value": model.pvalues[name],
    })


def contrast(model, weights):
    #t-test of a linear combination of coefficients, e.g. {"a": 1, "b": -1} for a - b
    r = np.zeros(len(model.params))
    for name, w in weights.items():
        r[model.params.index.get_loc(name)] = w
    t = model.t_test(r)
    lo, hi = np.asarray(t.conf_int()).ravel()
    return pd.Series({"effect": float(np.asarray(t.effect).ravel()[0]), "ci_low": lo, "ci_high": hi,
                      "p_value": float(np.asarray(t.pvalue).ravel()[0])})

## 2. How correlated is the wind across regions?

Weather systems cover hundreds of kilometres, so the regional wind indices move together.
This matters twice: it is the reason why regional wind must be controlled for (H3), and it tells us
whether Lithuania can be separated statistically from Latvia and Estonia. If the correlation between
Lithuania and its neighbours is above ~0.9, their separate coefficients would be unstable, so the
model uses the Baltic mean.

In [ ]:
regions = ["lt", "lv", "ee", "fi", "se", "no", "dk", "pl", "de"]
corr = df[[f"wind_cf_{r}" for r in regions]].corr()
corr.index = corr.columns = [r.upper() for r in regions]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(regions)), corr.columns)
ax.set_yticks(range(len(regions)), corr.index)
for i in range(len(regions)):
    for j in range(len(regions)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if corr.iloc[i, j] > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation of hourly ERA5 wind indices")
fig.tight_layout()
fig.savefig(FIG / "q1_wind_correlation.png", dpi=150)
plt.show()

print(f"Lithuania vs Latvia:  {corr.loc['LT', 'LV']:.2f}")
print(f"Lithuania vs Estonia: {corr.loc['LT', 'EE']:.2f}")

## 3. The supply curve: price against residual load

Residual load is demand minus wind and solar output, i.e. what the remaining power plants and
imports must cover. Plotting the price against it shows the shape of the merit order for each year.
This chart is descriptive only: it uses actual generation, which is itself affected by prices.

In [ ]:
d["residual_gw"] = d["residual_load_mw"] / 1000
edges = np.arange(np.floor(d["residual_gw"].min() * 4) / 4, d["residual_gw"].max() + 0.25, 0.25)
d["residual_bin"] = pd.cut(d["residual_gw"], edges)
curve = (d.groupby(["year", "residual_bin"], observed=True)["price"]
           .agg(["median", "count"]).reset_index())
curve = curve[curve["count"] >= 30]                     # skip bins with too few hours
curve["x"] = curve["residual_bin"].apply(lambda b: b.mid)

fig, ax = plt.subplots()
for year, g in curve.groupby("year"):
    ax.plot(g["x"], g["median"], marker="o", ms=3, label=str(year))
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Residual load: load − wind − solar (GW)")
ax.set_ylabel("Median price (€/MWh)")
ax.set_title("The Lithuanian supply curve by year")
ax.legend(title="Year")
fig.tight_layout()
fig.savefig(FIG / "q1_supply_curve.png", dpi=150)
plt.show()

## 4. H1 and H5 — the main regression

`price = β·wind_baltic + regional wind + solar + load forecast + gas + hydro + fixed effects + error`

Standard errors are Newey–West (HAC) with 48 lags, because errors in neighbouring hours are correlated.
H1 is tested with the full model (including regional wind); H5 is the hydro coefficient of the same model.

In [ ]:
m_base = smf.ols(f"price ~ wind_baltic + {CONTROLS}", data=d).fit(**HAC)
m_full = smf.ols(f"price ~ wind_baltic + wind_nordic + wind_continental + {CONTROLS}", data=d).fit(**HAC)

effects = pd.DataFrame({
    "+10 pp Baltic wind": effect(m_full, "wind_baltic", 10),
    "+10 pp Nordic wind": effect(m_full, "wind_nordic", 10),
    "+10 pp Continental wind": effect(m_full, "wind_continental", 10),
    "+100 W/m² solar radiation": effect(m_full, "solar_rad"),
    "+1 GW load forecast": effect(m_full, "load_fc_gw"),
    "+10 €/MWh gas": effect(m_full, "gas", 10),
    "+10 TWh Nordic hydro vs normal": effect(m_full, "hydro_twh", 10),
}).T
print(f"R² = {m_full.rsquared:.3f}, hours = {int(m_full.nobs):,}")
effects.round(3)

## 5. H3 — does regional wind matter?

Compare the Baltic wind coefficient without and with Nordic and Continental wind.
If weather systems are regional, the Baltic coefficient without regional wind also picks up
the effect of wind elsewhere and is too large in absolute value.

In [ ]:
h3 = pd.DataFrame({
    "Baltic wind, model without regional wind": effect(m_base, "wind_baltic", 10),
    "Baltic wind, model with regional wind": effect(m_full, "wind_baltic", 10),
    "Nordic wind": effect(m_full, "wind_nordic", 10),
    "Continental wind": effect(m_full, "wind_continental", 10),
}).T
h3.round(3)

## 6. H2 — has the wind effect grown over the years?

The Baltic wind coefficient is estimated separately for each year (interaction of wind with year).

In [ ]:
m_year = smf.ols(f"price ~ wind_baltic:C(year) + wind_nordic + wind_continental + {CONTROLS}",
                 data=d).fit(**HAC)
years = sorted(d["year"].unique())
by_year = pd.DataFrame({y: effect(m_year, f"wind_baltic:C(year)[{y}]", 10) for y in years}).T

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(by_year.index.astype(str), by_year["effect"],
            yerr=[by_year["effect"] - by_year["ci_low"], by_year["ci_high"] - by_year["effect"]],
            fmt="o", capsize=5)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("€/MWh per +10 pp Baltic wind")
ax.set_title("Effect of Baltic wind on the Lithuanian price, by year (95% CI)")
fig.tight_layout()
fig.savefig(FIG / "q1_wind_effect_by_year.png", dpi=150)
plt.show()

h2_test = contrast(m_year, {f"wind_baltic:C(year)[{years[-1]}]": 10, f"wind_baltic:C(year)[{years[0]}]": -10})
print(f"Change {years[0]} → {years[-1]}: {h2_test['effect']:.2f} €/MWh per +10 pp, p = {h2_test['p_value']:.4f}")
by_year.round(3)

## 7. H4 — does wind lower the price more when gas is expensive?

Wind and gas are centred on their means, so the interaction coefficient reads as
"how much the effect of +10 pp wind changes per +1 €/MWh of gas".

**Important:** the wind effect is allowed to differ by year (as in H2). Otherwise two trends would be
mixed up: gas was expensive in 2023, when there was little wind capacity, and cheaper later, when there
was much more. A model without year-specific wind effects would wrongly conclude that wind matters *less*
when gas is expensive. With them, the gas interaction is estimated from gas price changes *within* each year.
The table shows the implied effect of +10 pp wind in the latest year at low, median and high gas prices.

In [ ]:
d["wind_baltic_c"] = d["wind_baltic"] - d["wind_baltic"].mean()
d["gas_c"] = d["gas"] - d["gas"].mean()
m_gas = smf.ols(f"price ~ wind_baltic_c:C(year) + wind_baltic_c:gas_c + gas_c + wind_nordic + "
                f"wind_continental + solar_rad + load_fc_gw + hydro_twh + {FE}", data=d).fit(**HAC)

h4_inter = effect(m_gas, "wind_baltic_c:gas_c", 10)
print(f"Interaction: the effect of +10 pp wind changes by {h4_inter['effect']:.3f} €/MWh "
      f"per +1 €/MWh gas (p = {h4_inter['p_value']:.4f})")

levels = d["gas"].quantile([0.1, 0.5, 0.9])
last_year = years[-1]
at_gas = pd.DataFrame({
    f"{last_year}, gas = {g:.0f} €/MWh ({int(q * 100)}th percentile)": contrast(
        m_gas, {f"wind_baltic_c:C(year)[{last_year}]": 10, "wind_baltic_c:gas_c": 10 * (g - d["gas"].mean())})
    for q, g in levels.items()
}).T
at_gas.round(3)

## 8. Robustness checks

### 8.1 Coefficient stability
Controls are added one at a time. If the wind coefficient stays in the same range once the main
controls are in, an important omitted variable is less likely.

In [ ]:
steps = [
    ("no controls", ""),
    ("+ fixed effects", f" + {FE}"),
    ("+ load forecast", f" + load_fc_gw + {FE}"),
    ("+ gas", f" + load_fc_gw + gas + {FE}"),
    ("+ hydro", f" + load_fc_gw + gas + hydro_twh + {FE}"),
    ("+ solar", f" + load_fc_gw + gas + hydro_twh + solar_rad + {FE}"),
    ("+ regional wind", f" + load_fc_gw + gas + hydro_twh + solar_rad + wind_nordic + wind_continental + {FE}"),
]
stability = pd.DataFrame({
    label: effect(smf.ols(f"price ~ wind_baltic{rhs}", data=d).fit(**HAC), "wind_baltic", 10)
    for label, rhs in steps
}).T
stability.round(3)

### 8.2 Placebo test: wind one week later
Wind that will blow a week later cannot change today's price. If its coefficient is clearly
different from zero, the model is picking up seasonal patterns rather than the effect of wind.

In [ ]:
d_pl = d.dropna(subset=["wind_baltic_lead168"])
m_placebo = smf.ols(f"price ~ wind_baltic + wind_baltic_lead168 + wind_nordic + wind_continental + {CONTROLS}",
                    data=d_pl).fit(**HAC)
placebo = pd.DataFrame({
    "Baltic wind now": effect(m_placebo, "wind_baltic", 10),
    "Baltic wind one week later (placebo)": effect(m_placebo, "wind_baltic_lead168", 10),
}).T
placebo.round(3)

### 8.3 Dose–response
Instead of a straight line, the model gets one dummy per decile of Baltic wind (decile 1 = calmest
tenth of hours = reference). If wind causes lower prices, the price should fall step by step.

In [ ]:
d["wind_decile"] = pd.qcut(d["wind_baltic"], 10, labels=range(1, 11))
m_dose = smf.ols(f"price ~ C(wind_decile) + wind_nordic + wind_continental + {CONTROLS}", data=d).fit(**HAC)
dose = pd.DataFrame({k: effect(m_dose, f"C(wind_decile)[T.{k}]") for k in range(2, 11)}).T
mid = d.groupby("wind_decile", observed=True)["wind_baltic"].median()

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(mid.loc[dose.index], dose["effect"],
            yerr=[dose["effect"] - dose["ci_low"], dose["ci_high"] - dose["effect"]], fmt="o-", capsize=4)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Baltic wind, median of decile (% of capacity)")
ax.set_ylabel("Price vs calmest decile (€/MWh)")
ax.set_title("Dose–response: price by Baltic wind decile (95% CI)")
fig.tight_layout()
fig.savefig(FIG / "q1_dose_response.png", dpi=150)
plt.show()

monotonic = dose["effect"].reset_index(drop=True).corr(pd.Series(range(len(dose))), method="spearman")
print(f"Rank correlation between decile and effect: {monotonic:.2f} (−1 = perfectly decreasing)")

### 8.4 Before and after synchronisation with Continental Europe (9 February 2025)

In [ ]:
SYNC = pd.Timestamp("2025-02-09")
periods = {"before synchronisation": d[d["ts_local"] < SYNC], "after synchronisation": d[d["ts_local"] >= SYNC]}
sync = pd.DataFrame({
    name: effect(smf.ols(f"price ~ wind_baltic + wind_nordic + wind_continental + {CONTROLS}", data=part).fit(**HAC),
                 "wind_baltic", 10)
    for name, part in periods.items()
}).T
sync.round(3)

### 8.5 Instrumental variable (2SLS): the effect of 100 MW of actual wind output
Actual Lithuanian wind output is partly driven by the price (curtailment), so plain OLS on it is biased.
The ERA5 Lithuanian wind index is used as an instrument: it moves wind output but is not affected by the
price. Latvian/Estonian and regional wind are controlled for, so the instrument affects the price only
through Lithuanian wind output. A first-stage F statistic far above 10 means the instrument is strong.

In [ ]:
IV_CONTROLS = f"wind_lv_ee + wind_nordic + wind_continental + {CONTROLS}"
d_iv = d.dropna(subset=["wind_mw_100", "wind_lt", "wind_lv_ee"])

m_ols_mw = smf.ols(f"price ~ wind_mw_100 + {IV_CONTROLS}", data=d_iv).fit(**HAC)
m_iv = IV2SLS.from_formula(f"price ~ 1 + {IV_CONTROLS} + [wind_mw_100 ~ wind_lt]", data=d_iv).fit(
    cov_type="kernel", kernel="bartlett", bandwidth=48)

first_stage_f = m_iv.first_stage.diagnostics.loc["wind_mw_100", "f.stat"]
print(f"First-stage F statistic: {first_stage_f:,.0f}")
iv_table = pd.DataFrame({
    "OLS on actual wind output (biased by curtailment)": effect(m_ols_mw, "wind_mw_100"),
    "2SLS, ERA5 wind as instrument": effect(m_iv, "wind_mw_100"),
}).T
iv_table.round(3)

### 8.6 Median regression
Negative prices and price spikes pull the mean. A median (quantile) regression is less sensitive to them.

In [ ]:
m_median = smf.quantreg(f"price ~ wind_baltic + wind_nordic + wind_continental + {CONTROLS}", data=d).fit(
    q=0.5, max_iter=5000)
print(f"Median regression, +10 pp Baltic wind: {m_median.params['wind_baltic'] * 10:.2f} €/MWh "
      f"(OLS: {m_full.params['wind_baltic'] * 10:.2f} €/MWh)")

## 9. Summary of H1–H5

A hypothesis is supported when the estimate has the expected sign and p < 0.05.

In [ ]:
def verdict(ok):
    return "supported" if ok else "not supported"

h1 = effect(m_full, "wind_baltic", 10)
h5 = effect(m_full, "hydro_twh", 10)
regional_ok = any((effect(m_full, v)["effect"] < 0) and (effect(m_full, v)["p_value"] < 0.05)
                  for v in ["wind_nordic", "wind_continental"])
smaller = abs(m_full.params["wind_baltic"]) < abs(m_base.params["wind_baltic"])

summary = pd.DataFrame([
    ["H1", "Baltic wind lowers the price",
     f"{h1['effect']:.2f} €/MWh per +10 pp (p = {h1['p_value']:.4f})",
     verdict(h1["effect"] < 0 and h1["p_value"] < 0.05)],
    ["H2", "The wind effect has grown",
     f"{h2_test['effect']:.2f} €/MWh per +10 pp, {years[0]} → {years[-1]} (p = {h2_test['p_value']:.4f})",
     verdict(h2_test["effect"] < 0 and h2_test["p_value"] < 0.05)],
    ["H3", "Regional wind matters; Baltic effect overstated without it",
     f"Baltic {m_base.params['wind_baltic'] * 10:.2f} → {m_full.params['wind_baltic'] * 10:.2f} €/MWh per +10 pp",
     verdict(regional_ok and smaller)],
    ["H4", "Wind lowers the price more when gas is expensive",
     f"interaction {h4_inter['effect']:.3f} (p = {h4_inter['p_value']:.4f})",
     verdict(h4_inter["effect"] < 0 and h4_inter["p_value"] < 0.05)],
    ["H5", "More Nordic hydro than normal lowers the price",
     f"{h5['effect']:.2f} €/MWh per +10 TWh (p = {h5['p_value']:.4f})",
     verdict(h5["effect"] < 0 and h5["p_value"] < 0.05)],
], columns=["hypothesis", "statement", "estimate", "verdict"])

placebo_ok = placebo.loc["Baltic wind one week later (placebo)", "p_value"] >= 0.05
print(f"Placebo test passed: {placebo_ok}")
print(f"Dose–response monotonic (rank correlation): {monotonic:.2f}")
print(f"2SLS: {iv_table.iloc[1]['effect']:.2f} €/MWh per +100 MW of wind output (first-stage F = {first_stage_f:,.0f})")
summary.to_csv(FIG / "q1_summary.csv", index=False)
summary